```mermaid
flowchart TD
    A([START]) --> B[planner: sinh queries]
    B --> C[search_official_docs]
    B --> D[search_community_examples]

    C --> E[reviewer]
    D --> E

    E --> F{Đã đủ bằng chứng?}
    F -- Chưa đủ --> B
    F -- Đủ --> G[final_answer]
    G --> H([END])
```

In [ ]:
import operator
from typing_extensions import TypedDict, Annotated

In [ ]:
from langgraph.graph import StateGraph, START, END

In [ ]:
# ===== State =====
class ResearchState(TypedDict):
    task: str
    queries: list[str]
    facts: Annotated[list[str], operator.add]
    round: int
    verdict: str
    final_answer: str

In [ ]:
# ===== Nodes =====
def planner(state: ResearchState):
    """
    Lập kế hoạch cho mỗi vòng nghiên cứu.
    Vòng 1: hỏi cơ bản
    Vòng 2: đào sâu vào giới hạn / pitfalls
    """
    current_round = state.get("round", 0) + 1
    task = state["task"]

    if current_round == 1:
        queries = [
            f"{task} official documentation",
            f"{task} examples and community practices",
        ]
    else:
        queries = [
            f"{task} limitations and edge cases",
            f"{task} production pitfalls and trade-offs",
        ]

    return {
        "round": current_round,
        "queries": queries,
        "verdict": "",
    }

def search_official_docs(state: ResearchState):
    """
    Giả lập truy xuất từ docs chính thức.
    Trong thực tế bạn có thể gọi retriever / web tool / vector DB ở đây.
    """
    q = state["queries"][0]
    fact = f"[OFFICIAL] Tìm được thông tin từ docs cho query: {q}"
    return {"facts": [fact]}

def search_community_examples(state: ResearchState):
    """
    Giả lập truy xuất từ nguồn community.
    """
    q = state["queries"][1]
    fact = f"[COMMUNITY] Tìm được ví dụ thực tế cho query: {q}"
    return {"facts": [fact]}

def reviewer(state: ResearchState):
    """
    Đánh giá xem số fact đã đủ để kết luận chưa.
    """
    facts = state.get("facts", [])
    current_round = state["round"]

    # Rule minh hoạ:
    # - Nếu đã có >= 4 fact thì đủ
    # - Hoặc nếu đã qua 2 vòng thì chốt luôn
    if len(facts) >= 4 or current_round >= 2:
        answer = (
            f"Kết luận cho task: {state['task']}\n\n"
            "Các bằng chứng thu thập được:\n"
            + "\n".join(f"- {f}" for f in facts)
        )
        return {
            "verdict": "enough",
            "final_answer": answer,
        }

    return {
        "verdict": "need_more",
        "final_answer": "",
    }

In [ ]:
# ===== Conditional edge =====
def route_after_review(state: ResearchState):
    if state["verdict"] == "enough":
        return END
    return "planner"

In [ ]:
# ===== Build graph =====
builder = StateGraph(ResearchState)

builder.add_node("planner", planner)
builder.add_node("search_official_docs", search_official_docs)
builder.add_node("search_community_examples", search_community_examples)
builder.add_node("reviewer", reviewer)

builder.add_edge(START, "planner")
builder.add_edge("planner", "search_official_docs")
builder.add_edge("planner", "search_community_examples")
builder.add_edge("search_official_docs", "reviewer")
builder.add_edge("search_community_examples", "reviewer")
builder.add_conditional_edges("reviewer", route_after_review)

graph = builder.compile()

In [ ]:
# ===== Run =====
result = graph.invoke({
    "task": "LangChain vs LangGraph for production AI agents",
    "queries": [],
    "facts": [],
    "round": 0,
    "verdict": "",
    "final_answer": "",
})

print(result["final_answer"])